In [13]:
from datetime import datetime
from typing import Literal, Optional
import pybamm
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from netlist_utils import setup_circuit, make_lcapy_circuit
from battery import LiionBatteryPack, PybammBattery, CLCBattery
import vessim as vs

import nest_asyncio
nest_asyncio.apply()

In [69]:
class ExperimentController(vs.Controller):
    def __init__(self):
        super().__init__()

    def step(self, time: datetime, p_delta: float, e_delta: float, state: dict) -> None:
        self.set_parameters["policy:min_soc"] = 0.6 if (time.hour > 11) or time.hour < 2 and state["storage"]["soc"] >= 0.6 else 0.3
        self.set_parameters["policy:charge_power"] = 180 if (time.day == 12 and time.hour >= 10 and time.hour < 12) else 0

In [70]:
class MinSocPolicy(vs.MicrogridPolicy):
    def __init__(self, charge_power: Optional[float] = None, min_soc = 0):
        self.charge_power = charge_power if charge_power else 0.0
        self.min_soc = min_soc

    def apply(self, p_delta: float, duration: int, storage: Optional[vs.Storage] = None) -> float:
        energy_delta = p_delta * duration
        if storage and self.min_soc >= storage.soc() and p_delta <= 0:
            storage.update(0.0, duration)
        elif storage is not None:
            if self.charge_power:
                energy_delta -= storage.update(self.charge_power, duration)
            else:
                energy_delta -= storage.update(p_delta, duration)

        return energy_delta

    def state(self) -> dict:
        return {
            "charge_power": self.charge_power,
            "min_soc": self.min_soc
        }


In [71]:
def default_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.87 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model_type=pybamm.lithium_ion.SPMe,
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("Chen2020"),
        output_variables=["Current [A]", "Voltage [V]", "Power [W]", "Discharge energy [W.h]"],
    )
    liion_battery_pack = LiionBatteryPack(
        model_type=pybamm.lithium_ion.SPMe,
        netlist=setup_circuit(Np=np, Ns=ns, I=0.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("Chen2020"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [72]:

def degredation_battery_models(initial_soc, np, ns, step_size):
    simple_battery = vs.SimpleBattery(capacity=18.87 * np * ns, initial_soc=initial_soc)
    clc_battery = CLCBattery(number_of_cells=ns*np, initial_soc=initial_soc)
    pybamm_battery = PybammBattery(
        model_type=pybamm.lithium_ion.SPMe,
        options={
            "SEI": "solvent-diffusion limited",
            "SEI porosity change": "true",
            "lithium plating": "partially reversible",
            "lithium plating porosity change": "true",  # alias for "SEI porosity change"
            "particle mechanics": ("swelling and cracking", "swelling only"),
            "SEI on cracks": "true",
            "loss of active material": "stress-driven",
        },
        initial_soc=initial_soc,
        number_of_cells=ns*np,
        parameter_values=pybamm.ParameterValues("OKane2022"),
        output_variables=["Voltage [V]", "Current [A]", "Power [W]"]
    )
    liion_battery_pack = LiionBatteryPack(
        model_type=pybamm.lithium_ion.SPMe,
        options={
            "SEI": "solvent-diffusion limited",
            "SEI porosity change": "true",
            "lithium plating": "partially reversible",
            "lithium plating porosity change": "true",  # alias for "SEI porosity change"
            "particle mechanics": ("swelling and cracking", "swelling only"),
            "SEI on cracks": "true",
            "loss of active material": "stress-driven",
        },
        netlist=setup_circuit(Np=np, Ns=ns, I=0.0),
        step_size=step_size,
        initial_soc=initial_soc,
        parameter_values=pybamm.ParameterValues("OKane2022"),
    )
    return [simple_battery, clc_battery, pybamm_battery, liion_battery_pack]

In [ ]:
liion_battery = default_battery_models(step_size=10, np=4, ns=4, initial_soc=1)[3]
for i in range(10):
    liion_battery.update(-18.78 * 16, 10)
circuit = make_lcapy_circuit(liion_battery.netlist)
circuit.draw("../thesis/Images/pack_circuit.pdf", cpt_size=1, dpi=150, node_spacing=1.5, style="american")

In [ ]:
STEP_SIZE = 10
C_RATES_DISCHARGE = [0.1, 0.2, 0.3, 0.5, 1.0, 2.0]
for c_rate in C_RATES_DISCHARGE:
    power = -18.78 * c_rate * 16
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=1, np=4, ns=4, step_size=STEP_SIZE)[:4]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Discharge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 1.5:
        environment.run(until=(1 / c_rate) * 3800)
    else:
        environment.run(until=2 / 3 * 3800)

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_DISCHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 1},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_DISCHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:4]):
        df = pd.read_csv(f"results/{battery}_Discharge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    fig.update_yaxes(row=row, col=col, title="$\\text{State-of-Charge}$", ticksuffix = "%")
    showlegend = False
fig.write_image("../thesis/Images/discharging_experiment.pdf", "pdf")
fig.show()

In [ ]:
STEP_SIZE = 10
C_RATES_CHARGE = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
for c_rate in C_RATES_CHARGE:
    power = 18.78 * c_rate * 16
    environment = vs.Environment(sim_start="2022-07-03 09:00:00")
    for battery in default_battery_models(initial_soc=0, np=4, ns=4, step_size=STEP_SIZE)[:4]:
        monitor = vs.Monitor(outfile=f"results/{type(battery).__name__}_Charge{c_rate}C.csv")
        environment.add_microgrid(
            actors=[vs.Actor(name=f"MockActor:{type(battery).__name__}", signal=vs.MockSignal(value=power))],
            storage=battery,
            controllers=[monitor],
            step_size=STEP_SIZE,
        )
    if c_rate < 0.7:
        environment.run(until=(1 / c_rate) * 4200)
    else:
        environment.run(until=1 / 0.7 * 4200)

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_yaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=tuple(f"C-Rate {i}" for i in C_RATES_CHARGE))
fig.update_layout(
    width=800,
    height=800,
    margin={"r": 0, "l": 0, "t": 20, "b": 0},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=14,
    legend={"xanchor": "right", "x": 0.75},
)
fig.update_xaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
fig.update_yaxes(
    {"showline": True, "linewidth": 1, "linecolor": "black", "ticks":"outside"}
)
showlegend = True
batteries = ["SimpleBattery", "CLCBattery", "PybammBattery", "LiionBatteryPack"]
colors = ["Blue", "Red", "Green", "Purple"]
for i, c_rate in enumerate(C_RATES_CHARGE):
    col = i % 2 + 1
    row = i // 2 + 1
    for battery, color in zip(batteries, colors[:4]):
        df = pd.read_csv(f"results/{battery}_Charge{c_rate}C.csv")
        fig.add_trace(
            go.Scatter(x=[i * STEP_SIZE / 3600 for i in range(len(df.index))], y=df["storage.soc"] * 100, showlegend=showlegend, line_color=color, name=battery), row=row, col=col
        )
    fig.update_xaxes(row=row, col=col, title="", ticksuffix="h")
    fig.update_yaxes(row=row, col=col, title="$\\text{State-of-Charge}$", ticksuffix = "%")
    showlegend = False
fig.write_image("../thesis/Images/charging_experiment.pdf", "pdf")
fig.show()

In [78]:
STEP_SIZE = 30
environment = vs.Environment(sim_start="2021-06-11 00:00:00")
for battery in default_battery_models(initial_soc=0.7, np=4, ns=4, step_size=STEP_SIZE)[1:2]:
    environment.add_microgrid(
        actors=[
            vs.ComputingSystem(name=f"MockComputingSystem:{type(battery).__name__}", nodes=[vs.MockSignal(value=12), vs.MockSignal(value=5)], pue=1.3),
            vs.Actor(name=f"MockSolar:{type(battery).__name__}", signal=vs.HistoricalSignal(actual=pd.read_csv("data/solar_data.csv", index_col=0) * 0.6 * 0.17)),
        ],
        policy=MinSocPolicy(min_soc=0.6),
        storage=battery,
        controllers=[vs.Monitor(outfile=f"results/{type(battery).__name__}.csv"), ExperimentController()],
        step_size=STEP_SIZE,
    )

environment.run(until=48 * 3600)

2024-07-13 15:04:55.609 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockComputingSystem:CLCBattery" ...
2024-07-13 15:04:55.610 | INFO     | mosaik.scenario:start:311 - Starting "Actor" as "MockSolar:CLCBattery" ...
2024-07-13 15:04:55.611 | INFO     | mosaik.scenario:start:311 - Starting "Grid" as "Grid-0" ...
2024-07-13 15:04:55.611 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "Monitor-15" ...
2024-07-13 15:04:55.613 | INFO     | mosaik.scenario:start:311 - Starting "Controller" as "ExperimentController-2" ...
2024-07-13 15:04:55.614 | INFO     | mosaik.scenario:start:311 - Starting "Storage" as "Storage-0" ...
2024-07-13 15:04:55.615 | INFO     | mosaik.scenario:run:651 - Starting simulation.
100%|██████████| 172800/172800 [00:04<00:00, 35487.83steps/s]
2024-07-13 15:05:00.486 | INFO     | mosaik.scenario:run:708 - Simulation finished successfully.


In [80]:
battery = "CLCBattery"
df = pd.read_csv(f"results/{battery}.csv", index_col=0)
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.03)
# First plot
fig.add_trace(
    go.Scatter(x=df.index, y=df[f"MockSolar:{battery}.p"], line_color="green", name="solar"), row=1, col=1
)
line_dashes = ["dot", "dash"]
for i, col in enumerate(df.filter(regex="nodes").columns):
    fig.add_trace(
        go.Scatter(
            x=df.index, y=-df[col], line_color="red", line_dash=line_dashes[i], name=f"Node{i}",
        )
    )
fig.update_yaxes(title_text="<actor><br>power", row=1, col=1, ticksuffix=" W")

# Second plot
pos_delta = df["p_delta"].apply(lambda x: max(x, 0))
neg_delta = df["p_delta"].apply(lambda x: min(x, 0))
fig.add_trace(
    go.Scatter(x=pos_delta.index, y=pos_delta, fill="tozeroy", line_color="green", showlegend=False), row=2, col=1
)
fig.add_trace(
    go.Scatter(x=neg_delta.index, y=neg_delta, fill="tozeroy", line_color="red", showlegend=False), row=2, col=1
)
fig.update_yaxes(title_text="delta<br>power", row=2, col=1, ticksuffix=" W")

# Third plot
fig.add_trace(
    go.Scatter(x=df.index, y=df["storage.soc"]*100, fill="tozeroy", line_color="gray", name="SoC", legend="legend3"), row=3, col=1
)
fig.add_trace(
    go.Scatter(x=df.index, y=df["policy.min_soc"]*100, line_dash="dot", line_color="black", name="min. SoC", legend="legend3"), row=3, col=1,
)
fig.update_yaxes(title_text="battery<br>SoC", row=3, col=1, range=[0, 100], title_standoff=15, ticksuffix="%")

# Fourth plot
fig.add_trace(
    go.Scatter(x=df.index, y=-df["e_delta"] / STEP_SIZE, line_color="red", fill="tozeroy", name="grid power", legend="legend4"), row=4, col=1
)
fig.update_yaxes(title_text="grid<br>power", row=4, col=1, ticksuffix=" W")

fig.update_layout(
    width=500,
    height=400,
    margin={"r": 0, "l": 0, "t": 0.1, "b": 0},
    legend={"x": 0.7, "y": 1.02, "bgcolor": "rgba(0,0,0,0)"},
    legend2={"x": 0.7, "y": 0.73, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    legend3={"x": 0.17, "y": 0.51, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    legend4={"x": 0.25, "y": 0.25, "yanchor": "top", "bgcolor": "rgba(0,0,0,0)"},
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_size=12,
    xaxis_tickformat = '%d %B (%a)\n %Y'
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks="outside")
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", title_font_size=14)
fig.show()